<a href="https://colab.research.google.com/github/Marcin19721205/ProcessControl/blob/main/SettingsSIMCRules.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import math

def get_float(prompt, allow_empty=False):
    while True:
        s = input(prompt).strip().replace(",", ".")
        if allow_empty and s == "":
            return None
        try:
            return float(s)
        except ValueError:
            print("Błąd: podaj poprawną liczbę.")

def simc_tau_c_and_settings(k, theta, tau1, tau2=None, u0=None, ymax=None):
    if k == 0:
        raise ValueError("k nie może być równe 0.")
    if theta < 0:
        raise ValueError("theta musi być >= 0.")
    if tau1 <= 0:
        raise ValueError("tau1 musi być > 0.")
    if tau2 is not None and tau2 <= 0:
        raise ValueError("tau2 musi być > 0.")

    tau_c_min = theta                                  # SIMC tight control
    tau_c_default = theta                              # default recommendation
    tau_c_half = theta / 2.0 if theta > 0 else 0.0    # szybsze, ale mniej odporne

    Kc_min = None
    tau_c_max = None

    if u0 is not None and ymax is not None:
        if ymax == 0:
            raise ValueError("ymax nie może być równe 0.")
        Kc_min = abs(u0) / abs(ymax)                  # minimalny gain dla smooth control
        if Kc_min == 0:
            raise ValueError("Kc_min wyszedł 0.")
        tau_c_max = tau1 / (abs(k) * Kc_min) - theta  # z PDF
        if tau_c_max <= tau_c_min:
            print("\nUwaga: tau_c_max <= tau_c_min. "
                  "To znaczy, że podane u0/ymax dają zbyt restrykcyjny warunek smooth control.")

    def pi_settings(tau_c):
        Kp = (tau1 / (tau_c + theta)) / k
        Ti = min(tau1, 4.0 * (tau_c + theta))
        return Kp, Ti

    def pid_settings(tau_c):
        Kp, Ti = pi_settings(tau_c)
        Td = tau2
        return Kp, Ti, Td

    out = {
        "tau_c_min": tau_c_min,
        "tau_c_default": tau_c_default,
        "tau_c_half": tau_c_half,
        "Kc_min": Kc_min,
        "tau_c_max": tau_c_max,
        "PI_default": pi_settings(tau_c_default),
        "PI_half": pi_settings(tau_c_half) if theta > 0 else pi_settings(max(1e-9, tau_c_half)),
        "PID_default": pid_settings(tau_c_default) if tau2 is not None else None,
        "PID_half": pid_settings(tau_c_half) if (tau2 is not None and theta > 0) else (pid_settings(max(1e-9, tau_c_half)) if tau2 is not None else None),
    }

    if tau_c_max is not None and tau_c_max > 0:
        out["PI_smooth"] = pi_settings(tau_c_max)
        out["PID_smooth"] = pid_settings(tau_c_max) if tau2 is not None else None
    else:
        out["PI_smooth"] = None
        out["PID_smooth"] = None

    return out

k = get_float("Podaj k - process gain: ")
theta = get_float("Podaj theta - dead time [sec]: ")
tau1 = get_float("Podaj tau1 - process time constant [sec]: ")
tau2 = get_float("Podaj tau2 - second process time constant [sec] (ENTER jeśli brak, potrzebne do PID): ", allow_empty=True)

print("\nOpcjonalnie do wyznaczenia tau_c_max (smooth control) podaj:")
u0 = get_float("u0 - required input change to reject disturbance (ENTER jeśli brak): ", allow_empty=True)
ymax = get_float("ymax - maximum allowed output deviation (ENTER jeśli brak): ", allow_empty=True)

res = simc_tau_c_and_settings(k, theta, tau1, tau2=tau2, u0=u0, ymax=ymax)

print("\n" + "-" * 70)
print("SIMC - ZAKRES TAU_C")
print("-" * 70)
print(f"tau_c_min      = {res['tau_c_min']:.6f} s   # tight control, zalecenie SIMC")
print(f"tau_c_default  = {res['tau_c_default']:.6f} s   # domyślnie przyjmij tau_c = theta")
print(f"tau_c_half     = {res['tau_c_half']:.6f} s   # szybsze strojenie, mniej odporne")

if res["Kc_min"] is not None:
    print(f"Kc_min         = {res['Kc_min']:.6f}")
    print(f"tau_c_max      = {res['tau_c_max']:.6f} s   # smooth control")
else:
    print("tau_c_max      = brak   # podaj u0 i ymax, inaczej górnej granicy nie policzysz")

print("\n" + "-" * 70)
print("PI SETTINGS")
print("-" * 70)
Kp, Ti = res["PI_default"]
print(f"PI @ tau_c=theta      -> Kp = {Kp:.6f}, Ti = {Ti:.6f} s")

Kp, Ti = res["PI_half"]
print(f"PI @ tau_c=theta/2    -> Kp = {Kp:.6f}, Ti = {Ti:.6f} s")

if res["PI_smooth"] is not None:
    Kp, Ti = res["PI_smooth"]
    print(f"PI @ tau_c=tau_c_max  -> Kp = {Kp:.6f}, Ti = {Ti:.6f} s")

print("\n" + "-" * 70)
print("PID SETTINGS (SERIES FORM)")
print("-" * 70)

if tau2 is None:
    print("Brak tau2 -> nie da się policzyć Td zgodnie z oryginalną regułą SIMC.")
else:
    Kp, Ti, Td = res["PID_default"]
    print(f"PID @ tau_c=theta      -> Kp = {Kp:.6f}, Ti = {Ti:.6f} s, Td = {Td:.6f} s")

    Kp, Ti, Td = res["PID_half"]
    print(f"PID @ tau_c=theta/2    -> Kp = {Kp:.6f}, Ti = {Ti:.6f} s, Td = {Td:.6f} s")

    if res["PID_smooth"] is not None:
        Kp, Ti, Td = res["PID_smooth"]
        print(f"PID @ tau_c=tau_c_max  -> Kp = {Kp:.6f}, Ti = {Ti:.6f} s, Td = {Td:.6f} s")

print("-" * 70)

Podaj k - process gain: 0.7
Podaj theta - dead time [sec]: 2.77
Podaj tau1 - process time constant [sec]: 12.5
Podaj tau2 - second process time constant [sec] (ENTER jeśli brak, potrzebne do PID): 1

Opcjonalnie do wyznaczenia tau_c_max (smooth control) podaj:
u0 - required input change to reject disturbance (ENTER jeśli brak): 2
ymax - maximum allowed output deviation (ENTER jeśli brak): 3

----------------------------------------------------------------------
SIMC - ZAKRES TAU_C
----------------------------------------------------------------------
tau_c_min      = 2.770000 s   # tight control, zalecenie SIMC
tau_c_default  = 2.770000 s   # domyślnie przyjmij tau_c = theta
tau_c_half     = 1.385000 s   # szybsze strojenie, mniej odporne
Kc_min         = 0.666667
tau_c_max      = 24.015714 s   # smooth control

----------------------------------------------------------------------
PI SETTINGS
----------------------------------------------------------------------
PI @ tau_c=theta      